# Task 4.1 — Per-Ticker Signal Analysis
Analyze how each ticker's sentiment signal evolves over time and predicts returns.

In [ ]:
import json, warnings, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr, pearsonr
from numpy.polynomial.polynomial import polyfit

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_style("whitegrid")

# ── paths ──────────────────────────────────────────────────────────────────
BASE = Path("output")
SFT_PATH = BASE / "filing_signals_sft.jsonl"
TICKER_MAP_PATH = Path("../task_1/ticker_mapping.json")

# ── load SFT signals ──────────────────────────────────────────────────────
if not SFT_PATH.exists():
    # Fall back to base signals if SFT not yet generated
    FALLBACK = BASE / "filing_signals.jsonl"
    if FALLBACK.exists():
        print(f"WARNING: {SFT_PATH} not found. Falling back to {FALLBACK}")
        print("Run task_2_5 / task_2_6 to produce SFT-rescored signals.\n")
        _load_path = FALLBACK
    else:
        raise FileNotFoundError(
            f"Neither {SFT_PATH} nor {FALLBACK} found. "
            "Run task_2_5 and task_2_6 first."
        )
else:
    _load_path = SFT_PATH

records = [json.loads(l) for l in open(_load_path)]
df = pd.DataFrame(records)
df["filing_date"] = pd.to_datetime(df["filing_date"])
df.sort_values(["ticker", "filing_date"], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── load ticker mapping ───────────────────────────────────────────────────
with open(TICKER_MAP_PATH) as f:
    ticker_map = json.load(f)
# invert: ticker -> sub_sector
ticker_to_sector = {}
for sector, tickers in ticker_map.items():
    for t in tickers:
        ticker_to_sector[t] = sector

print(f"Total filings : {len(df):,}")
print(f"Unique tickers: {df['ticker'].nunique()}")
print(f"Date range    : {df['filing_date'].min().date()} to {df['filing_date'].max().date()}")
print(f"Source file   : {_load_path}")

## Per-Ticker Summary Statistics
For each ticker: filing count, mean/std signal, mean excess return, Spearman correlation with p-value, most common cohort, and sub-sector.

In [ ]:
MIN_FILINGS = 5  # minimum filings to compute correlation

def ticker_stats(g: pd.DataFrame) -> pd.Series:
    """Compute summary stats for one ticker's filing history."""
    n = len(g)
    mean_sig = g["signal"].mean()
    std_sig = g["signal"].std()
    mean_ret = g["ret_21d_excess"].mean()
    mode_cohort = g["cohort"].mode().iloc[0] if len(g) > 0 else "N/A"
    sector = g["sub_sector"].iloc[0] if "sub_sector" in g.columns else "unknown"

    if n >= MIN_FILINGS:
        rho, pval = spearmanr(g["signal"], g["ret_21d_excess"])
    else:
        rho, pval = np.nan, np.nan

    return pd.Series({
        "n_filings": n,
        "mean_signal": mean_sig,
        "std_signal": std_sig,
        "mean_ret_21d": mean_ret,
        "spearman_rho": rho,
        "spearman_p": pval,
        "mode_cohort": mode_cohort,
        "sub_sector": sector,
    })

ticker_df = df.groupby("ticker").apply(ticker_stats, include_groups=False).reset_index()
ticker_df.sort_values("spearman_rho", ascending=False, inplace=True)

# ── styled display ────────────────────────────────────────────────────────
def highlight_significant(row):
    """Highlight rows where Spearman is significant at p < 0.10."""
    if pd.notna(row["spearman_p"]) and row["spearman_p"] < 0.10:
        return ["background-color: #d4edda"] * len(row)
    return [""] * len(row)

fmt = {
    "mean_signal": "{:.4f}",
    "std_signal": "{:.4f}",
    "mean_ret_21d": "{:.4f}",
    "spearman_rho": "{:.3f}",
    "spearman_p": "{:.3f}",
}
styled = (
    ticker_df.style
    .apply(highlight_significant, axis=1)
    .format(fmt, na_rep="—")
)
display(styled)

n_sig_10 = (ticker_df["spearman_p"] < 0.10).sum()
n_sig_05 = (ticker_df["spearman_p"] < 0.05).sum()
n_pos = (ticker_df["spearman_rho"] > 0).sum()
n_neg = (ticker_df["spearman_rho"] < 0).sum()
n_valid = ticker_df["spearman_rho"].notna().sum()
print(f"\nTickers with >= {MIN_FILINGS} filings (valid correlations): {n_valid}")
print(f"Positive Spearman: {n_pos}  |  Negative: {n_neg}")
print(f"Significant at p<0.10: {n_sig_10}  |  p<0.05: {n_sig_05}")

## Per-Ticker Spearman Distribution
Distribution of per-ticker signal-return Spearman correlations across the universe.

In [ ]:
valid = ticker_df.dropna(subset=["spearman_rho"])

fig, ax = plt.subplots(figsize=(14, 8))
ax.hist(valid["spearman_rho"], bins=25, color="#5b8def", edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", linestyle="--", linewidth=1.2, label="zero")
ax.axvline(valid["spearman_rho"].mean(), color="#e74c3c", linestyle="-", linewidth=1.5,
           label=f'mean = {valid["spearman_rho"].mean():.3f}')

n_pos = (valid["spearman_rho"] > 0).sum()
n_neg = (valid["spearman_rho"] < 0).sum()
n_sig10 = (valid["spearman_p"] < 0.10).sum()
n_sig05 = (valid["spearman_p"] < 0.05).sum()

txt = (f"Positive: {n_pos}  |  Negative: {n_neg}\n"
       f"Significant p<0.10: {n_sig10}  |  p<0.05: {n_sig05}")
ax.text(0.02, 0.95, txt, transform=ax.transAxes, fontsize=12,
        verticalalignment="top", bbox=dict(boxstyle="round,pad=0.4",
        facecolor="white", edgecolor="#cccccc", alpha=0.9))

ax.set_xlabel("Spearman(signal, ret_21d_excess)", fontsize=13)
ax.set_ylabel("Number of Tickers", fontsize=13)
ax.set_title("Distribution of Per-Ticker Spearman Correlations", fontsize=15, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Signal Trajectory Over Time — Top Tickers
The 4 tickers with the highest positive Spearman and the 4 with the most negative Spearman, showing how their sentiment signal evolves filing-by-filing.

In [ ]:
COHORT_COLORS = {
    "very_negative": "#e74c3c",
    "negative": "#e67e22",
    "neutral": "#95a5a6",
    "positive": "#27ae60",
    "very_positive": "#2980b9",
}

# pick 4 best + 4 worst by Spearman (must have valid rho)
valid_sorted = valid.sort_values("spearman_rho", ascending=False)
top4 = valid_sorted.head(4)["ticker"].tolist()
bot4 = valid_sorted.tail(4)["ticker"].tolist()
selected_8 = top4 + bot4

fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharey=True)
axes = axes.flatten()

for idx, tkr in enumerate(selected_8):
    ax = axes[idx]
    sub = df[df["ticker"] == tkr].copy()
    row = ticker_df[ticker_df["ticker"] == tkr].iloc[0]

    # line
    ax.plot(sub["filing_date"], sub["signal"], color="#bdc3c7", linewidth=1, zorder=1)

    # scatter colored by cohort
    for _, r in sub.iterrows():
        c = COHORT_COLORS.get(r["cohort"], "#95a5a6")
        ax.scatter(r["filing_date"], r["signal"], color=c, s=40, zorder=2, edgecolors="white", linewidth=0.5)

    ax.axhline(0, color="black", linewidth=0.5, linestyle="--", alpha=0.5)
    rho_str = f"{row['spearman_rho']:.2f}" if pd.notna(row["spearman_rho"]) else "N/A"
    p_str = f"{row['spearman_p']:.2f}" if pd.notna(row["spearman_p"]) else "N/A"
    ax.set_title(f"{tkr} (rho={rho_str}, p={p_str})", fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.set_ylabel("Signal" if idx % 4 == 0 else "")

# legend
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=k)
           for k, c in COHORT_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=10, frameon=True)
fig.suptitle("Signal Trajectory — Top 4 (positive rho) and Bottom 4 (negative rho)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Signal vs Return — Per Ticker Scatter
Same 8 tickers: scatter of signal (x) vs 21-day excess return (y) with OLS fit line.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, tkr in enumerate(selected_8):
    ax = axes[idx]
    sub = df[df["ticker"] == tkr].dropna(subset=["signal", "ret_21d_excess"])
    x, y = sub["signal"].values, sub["ret_21d_excess"].values

    # scatter colored by cohort
    for _, r in sub.iterrows():
        c = COHORT_COLORS.get(r["cohort"], "#95a5a6")
        ax.scatter(r["signal"], r["ret_21d_excess"], color=c, s=45, zorder=2,
                   edgecolors="white", linewidth=0.5)

    # OLS fit line
    if len(x) >= 2:
        coeffs = np.polyfit(x, y, 1)
        xfit = np.linspace(x.min(), x.max(), 50)
        ax.plot(xfit, np.polyval(coeffs, xfit), color="#e74c3c", linewidth=1.5, linestyle="--",
                label=f"slope={coeffs[0]:.3f}")
        ax.legend(fontsize=8, loc="best")

    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.axvline(0, color="black", linewidth=0.5, alpha=0.4)

    row = ticker_df[ticker_df["ticker"] == tkr].iloc[0]
    rho_str = f"{row['spearman_rho']:.2f}" if pd.notna(row["spearman_rho"]) else "N/A"
    ax.set_title(f"{tkr} (rho={rho_str})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Signal" if idx >= 4 else "")
    ax.set_ylabel("ret_21d_excess" if idx % 4 == 0 else "")

fig.suptitle("Signal vs 21-Day Excess Return — Per Ticker", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Sector-Level Ticker Analysis
Aggregate Spearman statistics by sub-sector: mean correlation, fraction of positive-rho tickers, best and worst ticker per sector.

In [ ]:
def sector_summary(g: pd.DataFrame) -> pd.Series:
    """Aggregate per-ticker Spearman stats within a sector."""
    v = g.dropna(subset=["spearman_rho"])
    if len(v) == 0:
        return pd.Series({"n_tickers": len(g), "n_valid": 0,
                          "mean_rho": np.nan, "pct_positive": np.nan,
                          "best_ticker": "N/A", "best_rho": np.nan,
                          "worst_ticker": "N/A", "worst_rho": np.nan})
    best_idx = v["spearman_rho"].idxmax()
    worst_idx = v["spearman_rho"].idxmin()
    return pd.Series({
        "n_tickers": len(g),
        "n_valid": len(v),
        "mean_rho": v["spearman_rho"].mean(),
        "pct_positive": (v["spearman_rho"] > 0).mean() * 100,
        "best_ticker": v.loc[best_idx, "ticker"],
        "best_rho": v.loc[best_idx, "spearman_rho"],
        "worst_ticker": v.loc[worst_idx, "ticker"],
        "worst_rho": v.loc[worst_idx, "spearman_rho"],
    })

sector_df = ticker_df.groupby("sub_sector").apply(sector_summary, include_groups=False).reset_index()
sector_df.sort_values("mean_rho", ascending=False, inplace=True)

fmt_sector = {
    "mean_rho": "{:.3f}", "pct_positive": "{:.1f}%",
    "best_rho": "{:.3f}", "worst_rho": "{:.3f}",
}
display(sector_df.style.format(fmt_sector, na_rep="--"))

## Sentiment Momentum — Does Change in Signal Predict Returns?
Compute the filing-over-filing change in signal per ticker. Test whether tickers with *improving* sentiment outperform.

In [ ]:
# compute signal_change per ticker (current - previous filing)
df_sorted = df.sort_values(["ticker", "filing_date"]).copy()
df_sorted["signal_change"] = df_sorted.groupby("ticker")["signal"].diff()
momentum = df_sorted.dropna(subset=["signal_change", "ret_21d_excess"])

# overall Spearman
rho_all, p_all = spearmanr(momentum["signal_change"], momentum["ret_21d_excess"])
print(f"Overall Spearman(signal_change, ret_21d_excess): rho={rho_all:.4f}, p={p_all:.4f}")
print(f"  N = {len(momentum):,} filings with valid signal_change\n")

# per-sector Spearman
print("Per-Sector Sentiment Momentum Spearman:")
print("-" * 55)
for sector in sorted(momentum["sub_sector"].unique()):
    sub = momentum[momentum["sub_sector"] == sector]
    if len(sub) >= 10:
        rho_s, p_s = spearmanr(sub["signal_change"], sub["ret_21d_excess"])
        sig_marker = "*" if p_s < 0.10 else " "
        print(f"  {sector:30s}  rho={rho_s:+.4f}  p={p_s:.3f} {sig_marker}  (n={len(sub)})")
    else:
        print(f"  {sector:30s}  too few observations (n={len(sub)})")

# scatter plot
fig, ax = plt.subplots(figsize=(14, 8))
for sector, color in zip(sorted(momentum["sub_sector"].unique()),
                         sns.color_palette("tab10", n_colors=len(momentum["sub_sector"].unique()))):
    sub = momentum[momentum["sub_sector"] == sector]
    ax.scatter(sub["signal_change"], sub["ret_21d_excess"], alpha=0.5, s=25,
               label=sector, color=color, edgecolors="white", linewidth=0.3)

# OLS fit
x_m, y_m = momentum["signal_change"].values, momentum["ret_21d_excess"].values
coeffs = np.polyfit(x_m, y_m, 1)
xfit = np.linspace(x_m.min(), x_m.max(), 100)
ax.plot(xfit, np.polyval(coeffs, xfit), color="black", linewidth=2, linestyle="--",
        label=f"OLS (slope={coeffs[0]:.4f})")

ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Signal Change (current - previous filing)", fontsize=13)
ax.set_ylabel("21-Day Excess Return", fontsize=13)
ax.set_title(f"Sentiment Momentum vs Return (rho={rho_all:.3f}, p={p_all:.3f})",
             fontsize=15, fontweight="bold")
ax.legend(fontsize=9, loc="best", framealpha=0.9)
plt.tight_layout()
plt.show()

## Cross-Sectional IC Analysis
Standard quant-finance Information Coefficient: for each month with 10+ filings, rank tickers by their most recent signal and compute Spearman(rank, ret_21d_excess).

In [ ]:
# assign year-month to each filing
df["ym"] = df["filing_date"].dt.to_period("M")

ic_records = []
for ym, group in df.groupby("ym"):
    if len(group) < 10:
        continue
    # use the most recent signal per ticker in this month
    latest = group.sort_values("filing_date").groupby("ticker").last()
    if len(latest) < 10:
        continue
    rho, pval = spearmanr(latest["signal"], latest["ret_21d_excess"])
    ic_records.append({"period": str(ym), "ic": rho, "p": pval, "n": len(latest)})

ic_df = pd.DataFrame(ic_records)
ic_df["period_dt"] = pd.to_datetime(ic_df["period"])

mean_ic = ic_df["ic"].mean()
std_ic = ic_df["ic"].std()
t_stat = mean_ic / (std_ic / np.sqrt(len(ic_df))) if std_ic > 0 else np.nan

print(f"Cross-Sectional IC Analysis")
print(f"  Periods with 10+ filings: {len(ic_df)}")
print(f"  Mean IC  : {mean_ic:.4f}")
print(f"  Std IC   : {std_ic:.4f}")
print(f"  t-stat   : {t_stat:.2f}")
print(f"  % positive IC months: {(ic_df['ic'] > 0).mean()*100:.1f}%")

# plot
fig, ax = plt.subplots(figsize=(16, 7))
colors = ["#27ae60" if v > 0 else "#e74c3c" for v in ic_df["ic"]]
ax.bar(ic_df["period_dt"], ic_df["ic"], color=colors, width=25, alpha=0.8, edgecolor="white")
ax.axhline(mean_ic, color="#2c3e50", linewidth=2, linestyle="--",
           label=f"Mean IC = {mean_ic:.4f} (t={t_stat:.2f})")
ax.axhline(0, color="black", linewidth=0.5)

ax.set_xlabel("Filing Month", fontsize=13)
ax.set_ylabel("Cross-Sectional IC (Spearman)", fontsize=13)
ax.set_title("Monthly Cross-Sectional Information Coefficient", fontsize=15, fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## Ticker Ranking Heatmap
Mean signal per ticker per year (2015-2025). Rows sorted by sub-sector then mean signal. Red = negative sentiment, green = positive.

In [ ]:
df["year"] = df["filing_date"].dt.year

# pivot: mean signal per ticker per year
pivot = df.pivot_table(index="ticker", columns="year", values="signal", aggfunc="mean")

# sort rows by sub_sector then by overall mean signal within sector
pivot["sub_sector"] = pivot.index.map(lambda t: ticker_to_sector.get(t, "zzz_unknown"))
pivot["mean_signal"] = pivot.drop(columns=["sub_sector"]).mean(axis=1)
pivot.sort_values(["sub_sector", "mean_signal"], ascending=[True, False], inplace=True)
row_labels = [f"{t} ({pivot.loc[t, 'sub_sector'][:3]})" for t in pivot.index]
heat_data = pivot.drop(columns=["sub_sector", "mean_signal"])

fig, ax = plt.subplots(figsize=(16, max(12, len(heat_data) * 0.22)))
vmax = max(abs(heat_data.min().min()), abs(heat_data.max().max()))
sns.heatmap(
    heat_data, ax=ax, cmap="RdYlGn", center=0, vmin=-vmax, vmax=vmax,
    linewidths=0.3, linecolor="#f0f0f0", yticklabels=row_labels,
    cbar_kws={"label": "Mean Signal", "shrink": 0.6},
    annot=True, fmt=".2f", annot_kws={"fontsize": 6},
)
ax.set_xlabel("Year", fontsize=13)
ax.set_ylabel("")
ax.set_title("Ticker-Year Mean Sentiment Signal Heatmap", fontsize=15, fontweight="bold")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

## Category Contribution Per Ticker — Tornado Chart
For the 5 most bullish and 5 most bearish tickers (by mean signal), show which category scores drive the overall signal. Positive categories extend right, negative extend left.

In [ ]:
# expand category_scores into columns
cat_df = pd.json_normalize(df["category_scores"])
cat_df["ticker"] = df["ticker"].values

# mean category score per ticker
cat_means = cat_df.groupby("ticker").mean(numeric_only=True)
all_cats = sorted(cat_means.columns)

# pick 5 most bullish and 5 most bearish by overall mean signal
ticker_mean_sig = df.groupby("ticker")["signal"].mean().sort_values()
bearish_5 = ticker_mean_sig.head(5).index.tolist()
bullish_5 = ticker_mean_sig.tail(5).index.tolist()
tornado_tickers = bearish_5 + bullish_5

fig, axes = plt.subplots(2, 5, figsize=(22, 12), sharey=True)
axes_flat = axes.flatten()

for idx, tkr in enumerate(tornado_tickers):
    ax = axes_flat[idx]
    scores = cat_means.loc[tkr].sort_values()

    colors = ["#e74c3c" if v < 0 else "#27ae60" for v in scores]
    ax.barh(range(len(scores)), scores.values, color=colors, edgecolor="white", height=0.7)
    ax.set_yticks(range(len(scores)))
    ax.set_yticklabels(scores.index, fontsize=7)
    ax.axvline(0, color="black", linewidth=0.5)

    mean_sig_val = ticker_mean_sig[tkr]
    label = "BEARISH" if idx < 5 else "BULLISH"
    ax.set_title(f"{tkr} ({label})\nmean sig={mean_sig_val:.3f}", fontsize=10, fontweight="bold")
    if idx in (0, 5):
        ax.set_ylabel("Category")
    ax.set_xlabel("Mean Category Score" if idx >= 5 else "")

fig.suptitle("Category Contributions — 5 Most Bearish (left) vs 5 Most Bullish (right)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Key Drivers and Risk Flags Frequency
Across all filings, how often does each category appear as a key driver vs a risk flag?

In [ ]:
# count driver and risk occurrences per category
from collections import Counter

driver_counts = Counter()
risk_counts = Counter()

for _, row in df.iterrows():
    for d in row.get("key_drivers", []) or []:
        driver_counts[d] += 1
    for r in row.get("risk_flags", []) or []:
        risk_counts[r] += 1

# combine into DataFrame
all_categories = sorted(set(driver_counts.keys()) | set(risk_counts.keys()))
freq_df = pd.DataFrame({
    "category": all_categories,
    "driver_count": [driver_counts.get(c, 0) for c in all_categories],
    "risk_count": [risk_counts.get(c, 0) for c in all_categories],
})
freq_df["net"] = freq_df["driver_count"] - freq_df["risk_count"]
freq_df.sort_values("net", ascending=True, inplace=True)

fig, ax = plt.subplots(figsize=(14, max(8, len(freq_df) * 0.45)))
y_pos = range(len(freq_df))

# driver bars (right, green)
ax.barh(y_pos, freq_df["driver_count"], color="#27ae60", alpha=0.85, label="Key Driver", height=0.7)
# risk bars (left, red — plotted as negative)
ax.barh(y_pos, -freq_df["risk_count"], color="#e74c3c", alpha=0.85, label="Risk Flag", height=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(freq_df["category"], fontsize=10)
ax.axvline(0, color="black", linewidth=0.8)

# labels at bar ends
for i, (_, r) in enumerate(freq_df.iterrows()):
    if r["driver_count"] > 0:
        ax.text(r["driver_count"] + 5, i, str(int(r["driver_count"])), va="center", fontsize=8, color="#27ae60")
    if r["risk_count"] > 0:
        ax.text(-r["risk_count"] - 5, i, str(int(r["risk_count"])), va="center", fontsize=8,
                color="#e74c3c", ha="right")

ax.set_xlabel("Frequency Across All Filings", fontsize=13)
ax.set_title("Key Drivers (green, right) vs Risk Flags (red, left) by Category",
             fontsize=15, fontweight="bold")
ax.legend(fontsize=12, loc="lower right")
plt.tight_layout()
plt.show()

print("\nTop 5 most common KEY DRIVERS:")
for cat, cnt in driver_counts.most_common(5):
    print(f"  {cat:30s}  {cnt}")
print("\nTop 5 most common RISK FLAGS:")
for cat, cnt in risk_counts.most_common(5):
    print(f"  {cat:30s}  {cnt}")

## Summary Statistics and Export
Key findings from the per-ticker signal analysis, exported to `output/ticker_analysis_results.json`.

In [ ]:
valid_tickers = ticker_df.dropna(subset=["spearman_rho"])
n_valid = len(valid_tickers)
n_pos_rho = (valid_tickers["spearman_rho"] > 0).sum()
n_neg_rho = (valid_tickers["spearman_rho"] < 0).sum()
n_sig10 = (valid_tickers["spearman_p"] < 0.10).sum()
n_sig05 = (valid_tickers["spearman_p"] < 0.05).sum()
avg_rho = valid_tickers["spearman_rho"].mean()

# recompute momentum rho if needed
rho_momentum, p_momentum = spearmanr(momentum["signal_change"], momentum["ret_21d_excess"])

print("=" * 65)
print("  PER-TICKER SIGNAL ANALYSIS — KEY FINDINGS")
print("=" * 65)
print(f"  Total tickers with >= {MIN_FILINGS} filings : {n_valid}")
print(f"  Positive signal-return Spearman   : {n_pos_rho}/{n_valid} ({n_pos_rho/n_valid*100:.1f}%)")
print(f"  Negative signal-return Spearman   : {n_neg_rho}/{n_valid} ({n_neg_rho/n_valid*100:.1f}%)")
print(f"  Average per-ticker Spearman       : {avg_rho:.4f}")
print(f"  Significant at p<0.10             : {n_sig10}")
print(f"  Significant at p<0.05             : {n_sig05}")
print(f"  Sentiment momentum Spearman       : {rho_momentum:.4f} (p={p_momentum:.4f})")
print(f"  Mean cross-sectional IC           : {mean_ic:.4f} (t={t_stat:.2f})")
print("=" * 65)

# ── export ────────────────────────────────────────────────────────────────
export = {
    "meta": {
        "source": str(_load_path),
        "total_filings": len(df),
        "unique_tickers": int(df["ticker"].nunique()),
        "min_filings_for_corr": MIN_FILINGS,
        "date_range": [str(df["filing_date"].min().date()), str(df["filing_date"].max().date())],
    },
    "aggregate": {
        "n_valid_tickers": int(n_valid),
        "n_positive_rho": int(n_pos_rho),
        "n_negative_rho": int(n_neg_rho),
        "avg_spearman_rho": round(float(avg_rho), 4),
        "n_significant_p10": int(n_sig10),
        "n_significant_p05": int(n_sig05),
        "sentiment_momentum_rho": round(float(rho_momentum), 4),
        "sentiment_momentum_p": round(float(p_momentum), 4),
        "mean_cross_sectional_ic": round(float(mean_ic), 4),
        "ic_t_stat": round(float(t_stat), 2),
    },
    "per_ticker": [],
}

for _, row in ticker_df.iterrows():
    entry = {
        "ticker": row["ticker"],
        "sub_sector": row["sub_sector"],
        "n_filings": int(row["n_filings"]),
        "mean_signal": round(float(row["mean_signal"]), 4),
        "std_signal": round(float(row["std_signal"]), 4) if pd.notna(row["std_signal"]) else None,
        "mean_ret_21d": round(float(row["mean_ret_21d"]), 4),
        "spearman_rho": round(float(row["spearman_rho"]), 4) if pd.notna(row["spearman_rho"]) else None,
        "spearman_p": round(float(row["spearman_p"]), 4) if pd.notna(row["spearman_p"]) else None,
        "mode_cohort": row["mode_cohort"],
    }
    export["per_ticker"].append(entry)

# per_sector
export["per_sector"] = []
for _, row in sector_df.iterrows():
    export["per_sector"].append({
        "sub_sector": row["sub_sector"],
        "n_tickers": int(row["n_tickers"]),
        "n_valid": int(row["n_valid"]),
        "mean_rho": round(float(row["mean_rho"]), 4) if pd.notna(row["mean_rho"]) else None,
        "pct_positive": round(float(row["pct_positive"]), 1) if pd.notna(row["pct_positive"]) else None,
        "best_ticker": row["best_ticker"],
        "best_rho": round(float(row["best_rho"]), 4) if pd.notna(row["best_rho"]) else None,
        "worst_ticker": row["worst_ticker"],
        "worst_rho": round(float(row["worst_rho"]), 4) if pd.notna(row["worst_rho"]) else None,
    })

out_path = BASE / "ticker_analysis_results.json"
with open(out_path, "w") as f:
    json.dump(export, f, indent=2)
print(f"\nExported to {out_path} ({len(export['per_ticker'])} tickers, {len(export['per_sector'])} sectors)")